# RAPTOR 축소 계층 요약

원문을 범주별로 묶고 요약 노드를 직접 만들어 계층 검색을 실행합니다.

## 실행 환경 준비

고정한 Python 패키지를 확인합니다.

In [1]:
%pip install -q langchain==1.4.0 langchain-openai==1.6.1 python-dotenv==1.2.3 numpy==2.5.3
import sys
print("준비 완료:", sys.version.split()[0])

Note: you may need to restart the kernel to use updated packages.
준비 완료: 3.12.13


## .env 찾기

가까운 .env에서 키를 읽고 존재 여부만 표시합니다.

In [2]:
# 현재 폴더에서 가까운 .env를 찾되 비밀 값은 출력하지 않습니다.
import os, time
from dotenv import find_dotenv, load_dotenv
env_path=find_dotenv(usecwd=True)
load_dotenv(env_path)
print("OPENAI_API_KEY:","있음" if os.getenv("OPENAI_API_KEY") else "없음")

OPENAI_API_KEY: 있음


## 공통 데이터

Ch12의 축소 데이터와 Ch11 방식의 정답 문서 ID를 사용합니다.

In [3]:
# 배포 묶음에서도 같은 상대 경로를 찾도록 프로젝트 루트를 거슬러 올라갑니다.
import json
from pathlib import Path
import numpy as np
root=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/"labs").is_dir())
data_dir=root/"labs/part4/ch14-rag-patterns/data"
docs=json.loads((data_dir/"documents.json").read_text(encoding="utf-8"))
queries=json.loads((data_dir/"queries.json").read_text(encoding="utf-8"))
print("문서·질문:",len(docs),len(queries))

문서·질문: 12 8


## 모델과 임베딩

모델과 256차원 문서 벡터를 준비합니다.

In [4]:
# 256차원은 Ch12와 같은 비교 조건이며 행 하나가 문서 하나입니다.
from langchain.chat_models import init_chat_model
from langchain_openai import OpenAIEmbeddings
chat=init_chat_model("gpt-4.1-mini",model_provider="openai",temperature=0)
embedder=OpenAIEmbeddings(model="text-embedding-3-small",dimensions=256)
vectors=np.array(embedder.embed_documents([d["text"] for d in docs]))
print("임베딩 배열:",vectors.shape)

임베딩 배열: (12, 256)


## 검색·생성 함수

검색 점수, 토큰, 시간, 추정 비용을 같은 방식으로 기록합니다.

In [5]:
# 코사인 유사도는 내적을 두 벡터의 길이로 나누어 계산합니다.
from typing import Any
usage_log: list[dict[str, float | int]] = []

def search(question: str, k: int = 3) -> list[tuple[dict[str, Any], float]]:
    q=np.array(embedder.embed_query(question))
    scores=vectors@q/(np.linalg.norm(vectors,axis=1)*np.linalg.norm(q))
    # argsort 결과를 뒤집어 점수가 큰 문서부터 k개를 고릅니다.
    return [(docs[i],float(scores[i])) for i in np.argsort(scores)[::-1][:k]]
def ask(prompt: str) -> tuple[str, int, float]:
    started=time.perf_counter()
    msg=chat.invoke(prompt)
    usage=msg.usage_metadata or {}
    details=usage.get("input_token_details", {})
    input_tokens=usage.get("input_tokens", 0)
    cached_tokens=details.get("cache_read", 0)
    output_tokens=usage.get("output_tokens", 0)
    tokens=usage.get("total_tokens", 0)
    # 2026-09-10 gpt-4.1-mini 공개 단가로 추정하며 실제 청구액과 다를 수 있습니다.
    usd=((input_tokens-cached_tokens)*0.40+cached_tokens*0.10+output_tokens*1.60)/1_000_000
    usage_log.append({"input":input_tokens,"cached":cached_tokens,"output":output_tokens,"usd":usd})
    return msg.text,tokens,round((time.perf_counter()-started)*1000,1)

def show_hits(hits: list[tuple[dict[str, Any], float]]) -> list[tuple[str, float]]:
    return [(d["doc_id"],round(score,3)) for d,score in hits]

## 범주별 묶음

각 원문을 category 값으로 묶습니다. library·facility·account에 각각 네 건이 들어가는지 확인합니다.

In [6]:
# category가 같은 문서를 한 목록에 모읍니다.
groups={}
for doc in docs:
    groups.setdefault(doc["category"],[]).append(doc)
print({name:len(items) for name,items in groups.items()})

{'library': 4, 'facility': 4, 'account': 4}


## 요약 노드 직접 생성

범주별 원문을 요약하고 하위 문서 ID를 함께 저장합니다. 세 요약의 내용과 오프라인 생성 토큰·시간을 확인합니다.

In [7]:
# 요약마다 원문 ID를 함께 저장해 나중에 원문을 찾습니다.
summaries=[]
total_tokens=0
total_ms=0.0
for category,items in groups.items():
    source="\n".join(d["text"] for d in items)
    text,tokens,latency=ask(f"다음 {category} 규정의 공통 주제를 3문장으로 요약하세요.\n{source}")
    child_ids=[item["doc_id"] for item in items]
    summaries.append({"doc_id":f"SUM-{category}","category":category,"child_doc_ids":child_ids,"text":text})
    total_tokens+=tokens
    total_ms+=latency
print([(s["doc_id"],s["text"]) for s in summaries])
print("요약 토큰·ms:",total_tokens,round(total_ms,1))

[('SUM-library', '도서는 대출 기간과 연장, 반납 규정에 따라 이용할 수 있으며, 연체 시 대출 제한이 적용됩니다. 소장하지 않은 책은 희망 도서로 신청할 수 있고, 전자책은 모바일 앱에서 제한된 권수와 기간 내에 대출 가능합니다. 전반적으로 도서 대출과 반납, 신청 방법 및 전자책 이용에 관한 규정을 안내합니다.'), ('SUM-facility', '이 시설의 이용 규정은 예약, 결제, 운영 시간, 그리고 분실물 관리에 관한 안내를 포함합니다. 스터디룸은 최소 2명 이상이 예약 가능하며, 예약 후 10분 이내 입실하지 않으면 취소됩니다. 인쇄 비용은 학생증 잔액으로 결제하며, 자료실 운영 시간과 분실물 보관 기간도 명확히 정해져 있습니다.'), ('SUM-account', '이 계정 규정은 사용자 인증과 접근 권한 관리에 중점을 두고 있습니다. 비밀번호 재설정, 학생증 분실 시 조치, 그리고 휴학생과 졸업생의 자료 이용 제한을 명확히 규정하고 있습니다. 이를 통해 이용자의 신원 확인과 자료 이용 범위를 체계적으로 관리하고 있습니다.')]
요약 토큰·ms: 680 5423.4


## 계층 검색

질문과 가장 가까운 요약을 고른 뒤 그 범주의 원문만 검색합니다. facility 오선택과 하위 문서 ID를 확인합니다.

In [8]:
# 선택된 범주 안에서 원 질문과 가까운 문서 세 개를 고릅니다.
summary_vectors=np.array(embedder.embed_documents([s["text"] for s in summaries]))
question="휴학이나 졸업 뒤 이용 권한은 어떻게 달라지나요?"
q=np.array(embedder.embed_query(question))
scores=summary_vectors@q/(np.linalg.norm(summary_vectors,axis=1)*np.linalg.norm(q))
chosen=summaries[int(np.argmax(scores))]["category"]
chosen_summary=next(s for s in summaries if s["category"]==chosen)
leaf_hits=[(d,s) for d,s in search(question,k=len(docs)) if d["category"]==chosen][:3]
print("선택한 요약 범주:",chosen)
print("요약의 하위 문서:",chosen_summary["child_doc_ids"])
print("하위 문서:",show_hits(leaf_hits))

선택한 요약 범주: facility
요약의 하위 문서: ['FAC-001', 'FAC-002', 'FAC-003', 'FAC-004']
하위 문서: [('FAC-003', 0.246), ('FAC-002', 0.214), ('FAC-001', 0.194)]
